# Week 05 Lab — Greedy Algorithms

## Objectives
- Experimentally verify the success and failure conditions of greedy algorithms.
- Understand when greedy strategies produce optimal solutions and when they do not.

## Contents
- **A-1**: Coin Change (Greedy vs. DP)
- **A-2**: Fractional Knapsack
- **A-3**: Huffman Coding

---
## A-1: Coin Change

The **greedy strategy** for coin change: always pick the largest coin that fits.

This works for standard coin systems (where each coin is a multiple of the next smaller one), but **fails** for non-standard coin sets.

Let's see both cases.

In [ ]:
def coin_change_greedy(amount, coins):
    """Greedy approach: always pick the largest coin that fits.
    
    Args:
        amount: target amount
        coins: list of coin denominations
        
    Returns:
        list of coins used, or None if impossible
    """
    coins_sorted = sorted(coins, reverse=True)
    result = []
    remaining = amount
    for coin in coins_sorted:
        while remaining >= coin:
            result.append(coin)
            remaining -= coin
    return result if remaining == 0 else None


def coin_change_dp(amount, coins):
    """DP solution for comparison -- always finds the optimal answer.
    
    Args:
        amount: target amount
        coins: list of coin denominations
        
    Returns:
        list of coins used (minimum count), or None if impossible
    """
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    parent = [-1] * (amount + 1)
    for i in range(1, amount + 1):
        for c in coins:
            if c <= i and dp[i - c] + 1 < dp[i]:
                dp[i] = dp[i - c] + 1
                parent[i] = c
    if dp[amount] == float('inf'):
        return None
    result = []
    cur = amount
    while cur > 0:
        result.append(parent[cur])
        cur -= parent[cur]
    return result

### Case 1: Greedy Succeeds (Standard Coins)

With standard coins like [500, 100, 50, 10], each coin is a multiple of the next smaller one. The greedy approach always finds the optimal solution.

In [ ]:
print("=== Case 1: Standard coins ===")
coins1, amount1 = [500, 100, 50, 10], 1260
g1 = coin_change_greedy(amount1, coins1)
print(f"Coins: {coins1}, Amount: {amount1}")
print(f"Greedy: {g1} ({len(g1)} coins)")

print()
print("=== Case 1b: US coins ===")
coins1b, amount1b = [25, 10, 5, 1], 63
g1b = coin_change_greedy(amount1b, coins1b)
d1b = coin_change_dp(amount1b, coins1b)
print(f"Coins: {coins1b}, Amount: {amount1b}")
print(f"Greedy:  {g1b} ({len(g1b)} coins)")
print(f"Optimal: {d1b} ({len(d1b)} coins)")
print(f"Greedy is optimal: {len(g1b) == len(d1b)}")

### Case 2: Greedy Fails (Non-Standard Coins)

When coins do not have a divisibility relationship, greedy can give suboptimal results.

In [ ]:
failure_cases = [
    ([1, 3, 4], 6,  "coins={1,3,4}, amount=6"),
    ([7, 5, 1], 10, "coins={7,5,1}, amount=10"),
    ([6, 4, 1], 8,  "coins={6,4,1}, amount=8"),
    ([12, 9, 1], 18, "coins={12,9,1}, amount=18"),
]

for coins, amount, label in failure_cases:
    g = coin_change_greedy(amount, coins)
    d = coin_change_dp(amount, coins)
    is_optimal = len(g) == len(d) if g and d else g == d
    
    print(f"--- {label} ---")
    print(f"  Greedy:  {g} ({len(g)} coins)")
    print(f"  Optimal: {d} ({len(d)} coins)")
    if not is_optimal:
        print(f"  -> Greedy FAILS! Uses {len(g) - len(d)} extra coins.")
    else:
        print(f"  -> Greedy is optimal.")
    print()

### Discussion: Coin Change

1. What property must a coin system have for the greedy algorithm to always produce optimal results?
2. For the case coins = {1, 3, 4} and amount = 6: trace through the greedy algorithm step by step. Why does it pick 4+1+1 instead of 3+3?
3. What is the time complexity of the greedy approach vs. the DP approach?
4. Can you think of a coin system where greedy fails to find *any* solution (returns None), but DP succeeds?

### Exercise: Analyze Your Own Coin System

**TODO**: Create your own non-standard coin system where greedy fails, and verify with both approaches.

In [ ]:
# TODO: Define your own coin system and amount where greedy fails
my_coins = []   # Replace with your coin denominations
my_amount = 0   # Replace with a target amount

# Uncomment and run after filling in:
# g = coin_change_greedy(my_amount, my_coins)
# d = coin_change_dp(my_amount, my_coins)
# print(f"Coins: {my_coins}, Amount: {my_amount}")
# print(f"Greedy:  {g} ({len(g)} coins)")
# print(f"Optimal: {d} ({len(d)} coins)")

---
## A-2: Fractional Knapsack

In the **fractional knapsack** problem, items can be divided. The greedy strategy is:
1. Compute value-per-weight ratio for each item
2. Sort items by ratio in descending order
3. Greedily pick items with the highest ratio first
4. If an item does not fit entirely, take a fraction of it

This greedy approach **always** produces the optimal solution for fractional knapsack
(unlike the 0-1 knapsack, where items cannot be split).

In [ ]:
def fractional_knapsack(capacity, items):
    """Solve the fractional knapsack problem using a greedy approach.
    
    Args:
        capacity: maximum weight the knapsack can hold
        items: list of (name, weight, value) tuples
        
    Returns:
        (total_value, selected_items)
        selected_items: [(name, weight_taken, value_taken, ratio, is_fractional), ...]
    """
    # Sort by value/weight ratio (descending)
    sorted_items = sorted(items, key=lambda x: x[2] / x[1], reverse=True)

    total_value = 0.0
    remaining = capacity
    selected = []

    print(f"  Knapsack capacity: {capacity}kg")
    print(f"  Items sorted by value/weight ratio:")
    print(f"  {'Name':<10} {'Weight':>8} {'Value':>8} {'Ratio':>10}")
    print(f"  {'-'*38}")
    for name, weight, value in sorted_items:
        ratio = value / weight
        print(f"  {name:<10} {weight:>7.1f}kg {value:>7.0f} {ratio:>9.1f}/kg")

    print(f"\n  --- Greedy selection ---")

    for name, weight, value in sorted_items:
        if remaining <= 0:
            break

        ratio = value / weight

        if weight <= remaining:
            # Take the whole item
            selected.append((name, weight, value, ratio, False))
            total_value += value
            remaining -= weight
            print(f"  + {name}: take all {weight}kg "
                  f"(value {value:.0f}, remaining capacity {remaining:.1f}kg)")
        else:
            # Take a fraction
            fraction = remaining / weight
            partial_value = value * fraction
            selected.append((name, remaining, partial_value, ratio, True))
            total_value += partial_value
            print(f"  + {name}: take {fraction:.1%} ({remaining:.1f}kg) "
                  f"(value {partial_value:.0f}, remaining capacity 0.0kg)")
            remaining = 0

    return total_value, selected

In [ ]:
# Example 1: Basic knapsack
items1 = [
    ("Gold",     10.0, 600),
    ("Silver",   20.0, 500),
    ("Bronze",   30.0, 400),
    ("Jewel",     5.0, 300),
    ("Pottery",  15.0, 200),
]

print("=== Example 1: Basic Knapsack ===")
total, selected = fractional_knapsack(40, items1)
print(f"\n  Total value: {total:.0f}")
print(f"  Items taken:")
for name, w, v, r, frac in selected:
    marker = " (fractional)" if frac else ""
    print(f"    - {name}: {w:.1f}kg, value={v:.0f}{marker}")

### Fractional vs. 0-1 Knapsack Comparison

The fractional knapsack (greedy) always gives a value >= the 0-1 knapsack (DP/brute force), because splitting items gives more flexibility.

In [ ]:
def knapsack_01_bruteforce(capacity, items):
    """0-1 Knapsack via brute force (for comparison)."""
    n = len(items)
    best_value = 0
    best_selection = []
    for mask in range(1 << n):
        total_weight = 0
        total_value = 0
        selection = []
        for i in range(n):
            if mask & (1 << i):
                name, weight, value = items[i]
                total_weight += weight
                total_value += value
                selection.append(i)
        if total_weight <= capacity and total_value > best_value:
            best_value = total_value
            best_selection = selection
    return best_value, best_selection


# Compare on a small example
items2 = [
    ("A",  10.0,  60),
    ("B",  20.0, 100),
    ("C",  30.0, 120),
]
capacity = 50

print("=== Fractional vs. 0-1 Knapsack ===")
print(f"Items: {items2}")
print(f"Capacity: {capacity}kg")
print()

frac_value, _ = fractional_knapsack(capacity, items2)
print(f"\n  Fractional knapsack value: {frac_value:.0f}")

bf_value, bf_sel = knapsack_01_bruteforce(capacity, items2)
print(f"  0-1 knapsack value:        {bf_value:.0f}")
print(f"  0-1 selected: {[items2[i][0] for i in bf_sel]}")
print(f"\n  Fractional({frac_value:.0f}) >= 0-1({bf_value:.0f}): {frac_value >= bf_value}")

### Discussion: Fractional Knapsack

1. Why does the greedy approach (sort by value/weight) always produce the optimal solution for fractional knapsack?
2. Why does this greedy strategy NOT work for the 0-1 knapsack problem?
3. What is the time complexity of the fractional knapsack algorithm? What dominates the cost?
4. If two items have the same value/weight ratio, does it matter which one we pick first?

### Exercise: Custom Knapsack Problem

**TODO**: Create your own knapsack instance with at least 6 items and solve it.

In [ ]:
# TODO: Define your own items and capacity
my_items = [
    # ("name", weight, value),
]
my_capacity = 0  # Replace with your capacity

# Uncomment and run after filling in:
# total, selected = fractional_knapsack(my_capacity, my_items)
# print(f"\nTotal value: {total:.0f}")

---
## A-3: Huffman Coding

**Huffman coding** is a greedy algorithm for lossless data compression.

Greedy strategy: at each step, merge the two nodes with the **lowest frequency**.

This produces an optimal **prefix code** -- no code is a prefix of another, and
more frequent characters get shorter codes.

### Steps:
1. Build a frequency table from the input text
2. Create leaf nodes for each character and insert into a min-heap
3. Repeatedly extract the two lowest-frequency nodes and merge them
4. The final node is the root of the Huffman tree
5. Generate codes by traversing the tree (left = '0', right = '1')

In [ ]:
import heapq
import math
from collections import Counter


class HuffmanNode:
    """A node in the Huffman tree."""

    def __init__(self, char=None, freq=0, left=None, right=None):
        self.char = char       # Character (None for internal nodes)
        self.freq = freq       # Frequency
        self.left = left
        self.right = right

    def __lt__(self, other):
        """Compare by frequency for the min-heap."""
        return self.freq < other.freq

    def is_leaf(self):
        return self.left is None and self.right is None

In [ ]:
def build_frequency_table(text):
    """Build a character frequency table from text."""
    return dict(Counter(text).most_common())


def build_huffman_tree(freq_table):
    """Build a Huffman tree from a frequency table using a min-heap.
    
    Greedy choice: always merge the two lowest-frequency nodes.
    """
    heap = []
    for char, freq in freq_table.items():
        heapq.heappush(heap, HuffmanNode(char=char, freq=freq))

    # Handle single-character case
    if len(heap) == 1:
        node = heapq.heappop(heap)
        return HuffmanNode(freq=node.freq, left=node)

    step = 0
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)

        merged = HuffmanNode(
            freq=left.freq + right.freq,
            left=left,
            right=right,
        )
        heapq.heappush(heap, merged)

        step += 1
        left_label = repr(left.char) if left.is_leaf() else f"[{left.freq}]"
        right_label = repr(right.char) if right.is_leaf() else f"[{right.freq}]"
        print(f"  Step {step}: merge {left_label}({left.freq}) + "
              f"{right_label}({right.freq}) = [{merged.freq}]")

    return heapq.heappop(heap)


def generate_codes(root, prefix="", codes=None):
    """Traverse the Huffman tree to generate binary codes.
    
    Left edge = '0', Right edge = '1'.
    """
    if codes is None:
        codes = {}
    if root is None:
        return codes
    if root.is_leaf():
        codes[root.char] = prefix if prefix else "0"
        return codes
    generate_codes(root.left, prefix + "0", codes)
    generate_codes(root.right, prefix + "1", codes)
    return codes


def encode(text, codes):
    """Encode text using Huffman codes."""
    return "".join(codes[char] for char in text)


def decode(encoded_text, root):
    """Decode a bit string using the Huffman tree."""
    decoded = []
    current = root
    for bit in encoded_text:
        if bit == "0":
            current = current.left
        else:
            current = current.right
        if current.is_leaf():
            decoded.append(current.char)
            current = root
    return "".join(decoded)

In [ ]:
def print_tree(node, prefix="", is_left=True, is_root=True):
    """Print the Huffman tree visually."""
    if node is None:
        return
    if is_root:
        connector = ""
    elif is_left:
        connector = "|-- (0) "
    else:
        connector = "`-- (1) "
    if node.is_leaf():
        label = f"{repr(node.char)} [{node.freq}]"
    else:
        label = f"[{node.freq}]"
    print(f"  {prefix}{connector}{label}")
    if not is_root:
        new_prefix = prefix + ("|       " if is_left else "        ")
    else:
        new_prefix = prefix
    if not node.is_leaf():
        print_tree(node.left, new_prefix, True, False)
        print_tree(node.right, new_prefix, False, False)

### Example 1: Simple Text

Let's run through the complete Huffman coding process with a simple string.

In [ ]:
text1 = "abracadabra"
print(f"=== Huffman Coding: '{text1}' ===")

# Step 1: Frequency table
print("\n[Step 1] Frequency Table")
freq_table = build_frequency_table(text1)
print(f"  {'Char':<8} {'Freq':>6} {'Ratio':>8}")
print(f"  {'-'*24}")
for char, freq in freq_table.items():
    ratio = freq / len(text1) * 100
    print(f"  {repr(char):<8} {freq:>6} {ratio:>7.1f}%")

# Step 2: Build tree
print("\n[Step 2] Build Huffman Tree (greedy: merge two smallest)")
root = build_huffman_tree(freq_table)

# Visualize tree
print("\n[Step 2b] Tree Structure:")
print_tree(root)

# Step 3: Generate codes
print("\n[Step 3] Generated Codes")
codes = generate_codes(root)
sorted_codes = sorted(codes.items(), key=lambda x: (len(x[1]), x[1]))
print(f"  {'Char':<8} {'Code':<15} {'Length':>6} {'Freq':>6}")
print(f"  {'-'*38}")
for char, code in sorted_codes:
    print(f"  {repr(char):<8} {code:<15} {len(code):>6} {freq_table[char]:>6}")

# Step 4: Encode
print("\n[Step 4] Encoding")
encoded = encode(text1, codes)
print(f"  Original: {repr(text1)}")
print(f"  Encoded:  {encoded}")

# Step 5: Decode and verify
print("\n[Step 5] Decoding & Verification")
decoded = decode(encoded, root)
print(f"  Decoded:  {repr(decoded)}")
print(f"  Match:    {decoded == text1}")

# Step 6: Compression stats
print("\n[Step 6] Compression Statistics")
original_bits = len(text1) * 8
encoded_bits = len(encoded)
compression = (1 - encoded_bits / original_bits) * 100
num_chars = len(freq_table)
fixed_bits = math.ceil(math.log2(num_chars)) if num_chars > 1 else 1
fixed_total = len(text1) * fixed_bits

# Shannon entropy (theoretical lower bound)
entropy = sum(-f/len(text1) * math.log2(f/len(text1)) for f in freq_table.values())

print(f"  Original (ASCII 8-bit):    {original_bits:>6} bits")
print(f"  Fixed-length ({fixed_bits}-bit):      {fixed_total:>6} bits")
print(f"  Huffman coding:            {encoded_bits:>6} bits")
print(f"  Entropy lower bound:       {len(text1) * entropy:>6.0f} bits")
print(f"  Compression vs ASCII:      {compression:.1f}%")
print(f"  Avg bits/char (Huffman):   {encoded_bits/len(text1):.3f}")
print(f"  Entropy:                   {entropy:.3f}")

### Example 2: Skewed Frequency Distribution

Huffman coding is most effective when character frequencies are highly skewed.
Let's compare a uniform distribution vs. a skewed one.

In [ ]:
# Skewed distribution: lots of 'a', few of other characters
text_skewed = "aaaaaaaabbbbccdd"
print(f"=== Skewed text: '{text_skewed}' ===")
freq_skewed = build_frequency_table(text_skewed)
root_skewed = build_huffman_tree(freq_skewed)
codes_skewed = generate_codes(root_skewed)
encoded_skewed = encode(text_skewed, codes_skewed)

print(f"\n  Codes: {codes_skewed}")
print(f"  Original:  {len(text_skewed) * 8} bits (ASCII)")
print(f"  Huffman:   {len(encoded_skewed)} bits")
print(f"  Compression: {(1 - len(encoded_skewed) / (len(text_skewed) * 8)) * 100:.1f}%")

print()

# More uniform distribution
text_uniform = "abcdabcdabcdabcd"
print(f"=== Uniform text: '{text_uniform}' ===")
freq_uniform = build_frequency_table(text_uniform)
root_uniform = build_huffman_tree(freq_uniform)
codes_uniform = generate_codes(root_uniform)
encoded_uniform = encode(text_uniform, codes_uniform)

print(f"\n  Codes: {codes_uniform}")
print(f"  Original:  {len(text_uniform) * 8} bits (ASCII)")
print(f"  Huffman:   {len(encoded_uniform)} bits")
print(f"  Compression: {(1 - len(encoded_uniform) / (len(text_uniform) * 8)) * 100:.1f}%")

print()
print("Notice: skewed distributions compress much better!")

### Prefix Code Verification

A key property of Huffman codes is that they are **prefix-free**: no code is a prefix of another code. This allows unambiguous decoding without separators.

In [ ]:
def verify_prefix_free(codes):
    """Verify that a set of codes is prefix-free."""
    code_list = list(codes.values())
    for i in range(len(code_list)):
        for j in range(len(code_list)):
            if i != j and code_list[j].startswith(code_list[i]):
                return False, (code_list[i], code_list[j])
    return True, None


# Verify our Huffman codes
is_pf, conflict = verify_prefix_free(codes)
print(f"Codes for '{text1}': {codes}")
print(f"Prefix-free: {is_pf}")

# Show a non-prefix-free example for comparison
bad_codes = {'a': '0', 'b': '01', 'c': '1'}
is_pf2, conflict2 = verify_prefix_free(bad_codes)
print(f"\nBad codes: {bad_codes}")
print(f"Prefix-free: {is_pf2} (conflict: {conflict2})")
print("Why is this a problem? '01' could be 'ab' or 'b' -- ambiguous!")

### Discussion: Huffman Coding

1. Why does the greedy strategy (always merge the two lowest-frequency nodes) produce an optimal prefix code?
2. How does Huffman coding relate to Shannon entropy? Can it ever achieve compression better than entropy?
3. What is the time complexity of building a Huffman tree with n distinct characters?
4. In what real-world applications is Huffman coding used? (Hint: think about file formats like ZIP, JPEG, MP3)

### Exercise: Encode Your Own Text

**TODO**: Try encoding a longer text and observe the compression ratio.

In [ ]:
# TODO: Replace with your own text (try a sentence or paragraph)
my_text = ""  # Replace with your text

if my_text:
    freq = build_frequency_table(my_text)
    root = build_huffman_tree(freq)
    codes = generate_codes(root)
    encoded = encode(my_text, codes)
    decoded = decode(encoded, root)

    print(f"Text length: {len(my_text)} characters")
    print(f"Distinct characters: {len(freq)}")
    print(f"Original (ASCII): {len(my_text) * 8} bits")
    print(f"Huffman encoded:  {len(encoded)} bits")
    print(f"Compression:      {(1 - len(encoded) / (len(my_text) * 8)) * 100:.1f}%")
    print(f"Decode matches:   {decoded == my_text}")
else:
    print("Enter some text in the my_text variable above to encode.")

---
## Summary

| Algorithm | Greedy Strategy | Always Optimal? | Time Complexity |
|-----------|----------------|-----------------|------------------|
| Coin Change | Pick largest coin first | Only for standard coin systems | O(k) per coin type |
| Fractional Knapsack | Pick highest value/weight ratio | Yes (items can be split) | O(n log n) |
| Huffman Coding | Merge two lowest-frequency nodes | Yes (for prefix codes) | O(n log n) |

### Key Takeaways
- A greedy algorithm makes locally optimal choices at each step, hoping for a global optimum.
- Greedy works when the problem has the **greedy choice property** (a local optimum leads to a global optimum) and **optimal substructure**.
- When greedy fails (like non-standard coin change), we need techniques like dynamic programming.
- Greedy algorithms are typically faster and simpler than DP, so always check if greedy works before resorting to DP.